<a href="https://colab.research.google.com/github/Sav200512/sp500-rfm-segmentation-cw/blob/main/Task_1_and_2_cw_(20232749_W2151870).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Task 1.1 Ticker selection and data acquisition

In [29]:
import os, json, platform
import numpy as np
import pandas as pd
import yfinance as yf

In [30]:
SEED = 42
START, END = "2021-01-01", "2025-01-01"   # END is exclusive
BENCH = "^GSPC"

for folder in ["data/raw", "data/processed", "figures"]:
    os.makedirs(folder, exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66


In [31]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_df = table[0]

sp500_df["Yahoo_Symbol"] = sp500_df["Symbol"].str.replace(".", "-", regex=False)
sp500_df["Retrieved_On"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")

sp500_df[["Symbol", "Yahoo_Symbol", "Security", "GICS Sector", "Retrieved_On"]].to_csv(
    "data/raw/sp500_tickers.csv", index=False)

tickers = sp500_df["Yahoo_Symbol"].tolist()
print("Total tickers extracted:", len(tickers), "| retrieved:", sp500_df["Retrieved_On"].iloc[0])
sp500_df[["Symbol", "Security", "GICS Sector"]].head(10)

Total tickers extracted: 503 | retrieved: 2026-11-11 06:33 UTC


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


In [32]:
frames = []
for i in range(0, len(tickers), 50):
    batch = tickers[i:i + 50]
    d = yf.download(batch, start=START, end=END, auto_adjust=False,
                    progress=False, threads=True)
    frames.append(d)
    print(f"Batch {i // 50 + 1} done ({len(batch)} tickers)")

raw = pd.concat(frames, axis=1)
close, adj_close, volume = raw["Close"], raw["Adj Close"], raw["Volume"]

bench = yf.download(BENCH, start=START, end=END, auto_adjust=False, progress=False)
if isinstance(bench.columns, pd.MultiIndex):
    bench.columns = bench.columns.get_level_values(0)

Batch 1 done (50 tickers)
Batch 2 done (50 tickers)
Batch 3 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 4 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 5 done (50 tickers)
Batch 6 done (50 tickers)
Batch 7 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 8 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 9 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 10 done (50 tickers)
Batch 11 done (3 tickers)


In [33]:
close.to_csv("data/raw/close_daily.csv.gz")
adj_close.to_csv("data/raw/adj_close_daily.csv.gz")
volume.to_csv("data/raw/volume_daily.csv.gz")
bench.to_csv("data/raw/gspc_daily.csv")

run_log = {
    "download_time_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "start": START, "end_exclusive": END, "benchmark": BENCH,
    "yfinance": yf.__version__, "pandas": pd.__version__,
    "python": platform.python_version(),
    "tickers_requested": len(tickers),
}
json.dump(run_log, open("data/raw/run_log.json", "w"), indent=2)

no_data = [t for t in tickers if t not in adj_close.columns or adj_close[t].isna().all()]
print("Requested:", len(tickers))
print("Returned with some data:", len(tickers) - len(no_data))
print("No data:", no_data)
print("Shares: first date", adj_close.index.min().date(), "| last date", adj_close.index.max().date())
print("Benchmark: first date", bench.index.min().date(), "| last date", bench.index.max().date(), "| rows", len(bench))
print("Share rows:", len(adj_close), "| shape close/adj/vol:", close.shape, adj_close.shape, volume.shape)


Requested: 503
Returned with some data: 498
No data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Shares: first date 2021-01-04 | last date 2024-12-31
Benchmark: first date 2021-01-04 | last date 2024-12-31 | rows 1005
Share rows: 1005 | shape close/adj/vol: (1005, 503) (1005, 503) (1005, 503)


Task 1.2 Data validation and alignment

In [34]:
n_sessions = len(adj_close)
reasons = {}

for t, r in diag.iterrows():
    why = []
    if r["n_obs"] == 0:
        why.append("no price data in study period")
    elif r["n_obs"] < n_sessions:
        why.append(f"incomplete history: first price {r['first_valid'].date()} ({int(r['n_obs'])}/{n_sessions} sessions)")
    elif r["miss_close"] > 0 or r["miss_vol"] > 0:
        why.append("missing Close or Volume")
    if r["nonpos_price"] > 0:
        why.append("nonpositive price")
    if r["neg_vol"] > 0:
        why.append("negative volume")
    if why:
        reasons[t] = "; ".join(why)

exclusion_log = pd.DataFrame({"Ticker": list(reasons), "Reason": list(reasons.values())})
included = [t for t in cols if t not in reasons]

exclusion_log.to_csv("data/processed/exclusion_log.csv", index=False)
pd.Series(included, name="Ticker").to_csv("data/processed/included_shares.csv", index=False)

print("Included:", len(included), "| Excluded:", len(exclusion_log))
print(exclusion_log["Reason"].str.split(":").str[0].value_counts())
print("Shares with any zero-volume days (not excluded, check):", int((diag.loc[included, "zero_vol"] > 0).sum()))

Included: 488 | Excluded: 15
Reason
incomplete history               10
no price data in study period     5
Name: count, dtype: int64
Shares with any zero-volume days (not excluded, check): 2
